#### Noisy Quantum SVM - Spambase

In [87]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

Qiskit: 1.4.4
Aer: 0.17.2
QML: 0.8.4


In [88]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [89]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
# from imblearn.over_sampling import RandomOverSampler  # For optional balancing

In [90]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC, PegasosQSVC

In [91]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make",
    "word_freq_address",
    "word_freq_all",
    "word_freq_3d",
    "word_freq_our",
    "word_freq_over",
    "word_freq_remove",
    "word_freq_internet",
    "word_freq_order",
    "word_freq_mail",
    "word_freq_receive",
    "word_freq_will",
    "word_freq_people",
    "word_freq_report",
    "word_freq_addresses",
    "word_freq_free",
    "word_freq_business",
    "word_freq_email",
    "word_freq_you",
    "word_freq_credit",
    "word_freq_your",
    "word_freq_font",
    "word_freq_000",
    "word_freq_money",
    "word_freq_hp",
    "word_freq_hpl",
    "word_freq_george",
    "word_freq_650",
    "word_freq_lab",
    "word_freq_labs",
    "word_freq_telnet",
    "word_freq_857",
    "word_freq_data",
    "word_freq_415",
    "word_freq_85",
    "word_freq_technology",
    "word_freq_1999",
    "word_freq_parts",
    "word_freq_pm",
    "word_freq_direct",
    "word_freq_cs",
    "word_freq_meeting",
    "word_freq_original",
    "word_freq_project",
    "word_freq_re",
    "word_freq_edu",
    "word_freq_table",
    "word_freq_conference",
    "char_freq_;",
    "char_freq_(",
    "char_freq_[",
    "char_freq_!",
    "char_freq_$",
    "char_freq_#",
    "capital_run_length_average",
    "capital_run_length_longest",
    "capital_run_length_total",
    # finally the target label column:
    "label"
]

# --- 1. Load the Spambase Dataset ---
file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

In [92]:
# 2. Some basic processing
print(f"Original shape of Spambase data: {df.shape}") # Prints original dataset shape
df.drop_duplicates(inplace=True) # Remove duplicates
print(f"Shape after dropping duplicates: {df.shape}\n") # Then print again the new shape

Original shape of Spambase data: (4210, 58)
Shape after dropping duplicates: (4210, 58)



In [93]:
# Data Preparation

# 1. Split features and target
X = df.drop('label', axis=1)
y = df['label']

# 2. Train-test split FIRST (to avoid data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)
# This gives you ~2947 training, ~1263 test samples

print(f"Full training set: {X_train.shape[0]} samples")
print(f"Full test set: {X_test.shape[0]} samples\n")

Full training set: 2947 samples
Full test set: 1263 samples



In [94]:
# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use same scaler

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)


In [95]:
# --- Feature Selection ---
print("--- Feature Selection ---")
THRESH = 0.9

# Calculate correlation matrix on the SCALED TRAINING data
corr_matrix_train = X_train_scaled_df.corr().abs()

# Get the upper triangle of the correlation matrix
upper_triangle = corr_matrix_train.where(np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool))

# Find features with correlation greater than the threshold
columns_to_drop = set()
for column in upper_triangle.columns:
    high_corr_partners = upper_triangle.index[upper_triangle[column] > THRESH].tolist()
    if high_corr_partners:
        for partner in high_corr_partners:
            # IMPORTANT: Check correlation with the TRAINING target variable
            corr_main_vs_target = y_train.corr(X_train_scaled_df[column])
            corr_partner_vs_target = y_train.corr(X_train_scaled_df[partner])
            
            print(f"Found pair: ('{column}', '{partner}') with correlation > {THRESH}")
            if abs(corr_main_vs_target) < abs(corr_partner_vs_target):
                columns_to_drop.add(column)
                print(f"-> Dropping '{column}' (weaker correlation with target)")
            else:
                columns_to_drop.add(partner)
                print(f"-> Dropping '{partner}' (weaker correlation with target)")

to_drop_final = sorted(list(columns_to_drop))
print(f"\nTotal features to drop ({len(to_drop_final)}): {to_drop_final}")

# Drop the identified columns from both training and test sets
X_train_selected = X_train_scaled_df.drop(columns=to_drop_final)
X_test_selected = X_test_scaled_df.drop(columns=to_drop_final)

print(f"\nOriginal number of features: {X_train.shape[1]}")
print(f"Number of features after selection: {X_train_selected.shape[1]}\n")

--- Feature Selection ---
Found pair: ('word_freq_415', 'word_freq_857') with correlation > 0.9
-> Dropping 'word_freq_415' (weaker correlation with target)

Total features to drop (1): ['word_freq_415']

Original number of features: 57
Number of features after selection: 56



In [96]:
# PCA
n_components = 4
pca = PCA(n_components=n_components, random_state=42)

# Fit on the selected training data and transform both sets
X_train_pca = pca.fit_transform(X_train_selected)
X_test_pca = pca.transform(X_test_selected)

print(f"Shape after PCA (Train): {X_train_pca.shape}")
print(f"Shape after PCA (Test):  {X_test_pca.shape}")

Shape after PCA (Train): (2947, 4)
Shape after PCA (Test):  (1263, 4)


##### Noise Simulation Setup

In [97]:
# Quantum Kernel Implementation with Noise
# Create noise model
p_error = 0.04  # 4% depolarizing error for 1-qubit gates
depolarizing_prob_2q = 2 * p_error  # 8% for 2-qubit gates

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(p_error, 1), ['u1', 'u2', 'u3', 'rx', 'ry', 'rz', 'id'])
noise_model.add_all_qubit_quantum_error(depolarizing_error(depolarizing_prob_2q, 2), ['cx', 'cz', 'ecr'])

print(f"Depolarizing noise: {p_error*100}% (1q), {p_error*200}% (2q)\n")

Depolarizing noise: 4.0% (1q), 8.0% (2q)



In [98]:
# Noisy backend
noisy_backend = AerSimulator(
    noise_model=noise_model,
    seed_simulator=12345,
)

In [99]:
# Noisy Sampler with high shots
noise_sampler = AerSampler.from_backend(
    backend = noisy_backend,
    default_shots = 256 
)
print("Noisy Sampler created !")

Noisy Sampler created !


In [100]:
# Transpilation pass manager
pm = generate_preset_pass_manager(optimization_level=1, backend=noisy_backend)
print("Noisy Sampler and pass manager ready!")

Noisy Sampler and pass manager ready!


##### Quantum Kernel Implementation

In [101]:
# Feature map setup
feature_dim = n_components
fm = ZZFeatureMap(feature_dimension=feature_dim, reps=1, entanglement='linear')

# Fidelity with noisy sampler and transpilation
fidelity = ComputeUncompute(sampler=noise_sampler, pass_manager=pm)

# Noisy quantum kernel
noisy_qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=fm)

##### QSVC Implementation

In [ ]:
print("="*70)
print(">>> PHASE 1: STANDARD QSVC (Subset 500)")
print("="*70)

# Subset for QSVC
X_train_subset, _, y_train_subset, _ = train_test_split(
    X_train_pca, y_train,
    train_size=500,
    stratify=y_train,
    random_state=42
)

qsvc_subset = QSVC(quantum_kernel=noisy_qkernel, C=1.0)
start_time_sub = time.time()
qsvc_subset.fit(X_train_subset, y_train_subset)
train_time_sub = time.time() - start_time_sub

# Predictions
y_train_pred_sub = qsvc_subset.predict(X_train_subset)
y_test_pred_sub = qsvc_subset.predict(X_test_pca)

train_accuracy_sub = accuracy_score(y_train_subset, y_train_pred_sub)
test_accuracy_sub = accuracy_score(y_test, y_test_pred_sub)
generalization_gap_sub = abs(train_accuracy_sub - test_accuracy_sub)

print(f"Samples Used: {X_train_subset.shape[0]}")
print(f"Training Accuracy: {train_accuracy_sub:.4f}")
print(f"Test Accuracy: {test_accuracy_sub:.4f}")
print(f"Training Time: {train_time_sub:.2f} seconds\n")

>>> PHASE 1: STANDARD QSVC (Subset 500)


##### PegasosQSVC

In [ ]:
# ==========================================
# EXPERIMENT B: PEGASOS QSVC (Full Dataset)
# ==========================================
print("="*70)
print(f">>> PHASE 2: PEGASOS QSVC (Full {X_train_pca.shape[0]} Samples)")
print("="*70)

# Reset index for Pegasos
y_train_reset = y_train.reset_index(drop=True)

pegasos_qsvc = PegasosQSVC(
    quantum_kernel=noisy_qkernel,
    C=1.0,
    num_steps=1000
)
start_time_peg = time.time()
pegasos_qsvc.fit(X_train_pca, y_train_reset)  # Full training set
train_time_peg = time.time() - start_time_peg

# Predictions
y_train_pred_peg = pegasos_qsvc.predict(X_train_pca)
y_test_pred_peg = pegasos_qsvc.predict(X_test_pca)

train_accuracy_peg = accuracy_score(y_train, y_train_pred_peg)
test_accuracy_peg = accuracy_score(y_test, y_test_pred_peg)
generalization_gap_peg = abs(train_accuracy_peg - test_accuracy_peg)

print(f"Samples Used: {X_train_pca.shape[0]}")
print(f"Training Accuracy: {train_accuracy_peg:.4f}")
print(f"Test Accuracy: {test_accuracy_peg:.4f}")
print(f"Training Time: {train_time_peg:.2f} seconds")


>>> PHASE 2: PEGASOS QSVC (Scalability Solution - 294 Samples)
--- PegasosQSVC Evaluation (294 Samples) ---
Training Accuracy:    0.6020
Test Accuracy:        0.5984
Generalization Gap:   0.0036
Training Time:        643.34 seconds

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.60      1.00      0.75        76
           1       0.00      0.00      0.00        51

    accuracy                           0.60       127
   macro avg       0.30      0.50      0.37       127
weighted avg       0.36      0.60      0.45       127



##### Model Evaluation

In [ ]:
print("\n" + "="*70)
print("MODEL EVALUATION SUMMARY")
print("="*70)

print(f"{'Metric':<25} | {'QSVC (Subset 500)':<20} | {'PegasosQSVC (Full)':<20}")
print("-" * 70)
print(f"{'Samples Used':<25} | {X_train_pca.shape[0]:<20} | {len(X_train_pca):<20}")
print(f"{'Training Time (s)':<25} | {train_time_sub:<20.2f} | {train_time_peg:<20.2f}")
print(f"{'Training Accuracy':<25} | {train_accuracy_sub:<20.4f} | {train_accuracy_peg:<20.4f}")
print(f"{'Test Accuracy':<25} | {test_accuracy_sub:<20.4f} | {test_accuracy_peg:<20.4f}")
print(f"{'Generalization Gap':<25} | {generalization_gap_sub:<20.4f} | {generalization_gap_peg:<20.4f}")
print("="*70)


MODEL EVALUATION SUMMARY
Metric                    | QSVC (Subset 500)    | PegasosQSVC (Full)  
----------------------------------------------------------------------
Samples Used              | 294                  | 294                 
Training Time (s)         | 2163.04              | 643.34              
Training Accuracy         | 0.9524               | 0.6020              
Test Accuracy             | 0.5591               | 0.5984              
Generalization Gap        | 0.3933               | 0.0036              
